# 03 · Compare full-data ML candidates (validation only)

This notebook runs Phase 6 corrected-target after the original PhysioNet files have been prepared into patient-separated train, validation and test CSVs. It trains a new logistic-regression model and a histogram gradient-boosted model using all training rows by default, then compares them on the validation set only.

**Important:** the comparison code does not read `test.csv`. Select the candidate using validation evidence first; run one final test evaluation later. This experiment uses the corrected onset-within-six-hours target. These are research artifacts, not ICU admission predictions or clinical recommendations.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd

CWD = Path.cwd().resolve()
candidates = [CWD, CWD / 'ml-service', CWD.parent / 'ml-service', CWD.parent.parent / 'ml-service']
ML_ROOT = next((candidate for candidate in candidates if (candidate / 'pyproject.toml').exists() and (candidate / 'src' / 'bedallocation_ml').exists()), None)
if ML_ROOT is None:
    raise FileNotFoundError('Could not locate ml-service. Open Jupyter from the repository or ml-service folder.')
print('ML root:', ML_ROOT)
print('Prepared data exists:', (ML_ROOT / 'data' / 'processed_onset' / 'train.csv').exists())

## Train and compare

This default command uses **all available training rows** and 100 boosting iterations. It can take several minutes and may use considerable memory on a laptop. If you need a quick code-path smoke test only, set `QUICK_SMOKE_TEST = True`; such capped results must not be confused with the full-data experiment.

In [ ]:
QUICK_SMOKE_TEST = False
command = [
    sys.executable, '-m', 'bedallocation_ml.compare_models',
    '--data-dir', 'data/processed_onset',
    '--artifact-dir', 'artifacts',
]
if QUICK_SMOKE_TEST:
    command += ['--max-train-rows', '250000', '--hgb-max-iter', '30']
subprocess.run(command, cwd=ML_ROOT, check=True)

## Review the validation report

In [ ]:
report_path = ML_ROOT / 'artifacts' / 'sepsis_onset_model_comparison_validation.json'
report = json.loads(report_path.read_text(encoding='utf-8'))
rows = []
for name, model in report['models'].items():
    metrics = model['validation_metrics']
    rows.append({
        'model': name,
        'train_rows_used': report['train_rows_used'],
        'validation_PR_AUC_average_precision': metrics['average_precision_pr_auc'],
        'validation_ROC_AUC': metrics['roc_auc'],
        'precision_at_0.5': metrics['precision_at_threshold'],
        'recall_at_0.5': metrics['recall_at_threshold'],
        'F1_at_0.5': metrics['f1_at_threshold'],
        'specificity_at_0.5': (metrics['confusion_matrix_labels_0_then_1'][0][0] / sum(metrics['confusion_matrix_labels_0_then_1'][0])) if sum(metrics['confusion_matrix_labels_0_then_1'][0]) else 0,
        'training_seconds': model['training_seconds'],
    })
comparison = pd.DataFrame(rows).sort_values('validation_PR_AUC_average_precision', ascending=False)
display(comparison)
print('Held-out test split used:', report['test_split_used'])
print('Validation ranking only:', report['validation_rank_by_average_precision'])
print('Clinical status:', report['clinical_status'])

## Before selecting a candidate

- Do not choose only by ROC-AUC or accuracy. Positive target prevalence is around 1.9%, so inspect average precision, precision, recall, flagged fraction and false-positive burden together.
- Threshold 0.5 is only a comparison point, not a clinical cutoff. Balanced class weights can make probability scores uncalibrated.
- Hourly row-level metrics may count several rows per patient; patient/episode-level analysis is needed before any real-world consideration.
- The corrected target estimates whether inferred annotated sepsis onset is in the next six hourly positions. It excludes post-onset rows and does not predict ICU admission or establish clinical urgency.
- Never connect these models to the live queue or allocation workflow without independent clinical review and validation.